Stochastic Model v3

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import numpy as np
from numpy import random


#SEPERATE POV FOR SPEED? MISSCHIEN VOOR DE PRESENTATIE EVEN HARDCODDEN
#RHO GRAFIEKEN MAKEN ZOALS EDOARDE HEEFT
#MISSCHIEN EEN HOGERE AANTAL MENSEN VOOR TIJDENS DE PRESENTATIE


Discretize the shape. Then use that to check if in boundary or not. For now we will just use simple shapes.

Should we use a different FOV for the speed interactions??


Squares are defined as follows:

square = [$x_{0}$,$x_{1}$,$x_{2}$,$x_{3}$]

![SNOWFALL](square.png)

In [ ]:
class Boundary:
    def __init__(self, corners):
        """corners: [bottom-left, bottom-right, top-left, top-right]"""
        self.corners = corners

    def contains(self, x, y):
        c = self.corners
        return c[0][0] < x < c[3][0] and c[0][1] < y < c[3][1]

    def draw(self, ax, color='black'):
        c = self.corners
        ax.plot([c[0][0], c[2][0]], [c[0][1], c[2][1]], color)
        ax.plot([c[1][0], c[3][0]], [c[1][1], c[3][1]], color)
        ax.plot([c[2][0], c[3][0]], [c[2][1], c[2][1]], color)
        ax.plot([c[0][0], c[1][0]], [c[0][1], c[1][1]], color)


class Exit(Boundary):
    def __init__(self, corners, target):
        super().__init__(corners)
        self.target = target

    def draw(self, ax, color='green'):
        c = self.corners
        ax.plot([c[1][0], c[3][0]], [c[1][1], c[3][1]], color, linewidth=2)

    def angle_to(self, x, y):
        return np.arctan2(self.target[1] - y, self.target[0] - x) % (2 * np.pi)

    def is_exiting(self, x, y):
        c = self.corners
        return x > c[1][0] and c[0][1] <= y <= c[2][1]


class Obstacle:
    def __init__(self, corners):
        """corners: [bottom-left, bottom-right, top-left, top-right]"""
        self.corners = corners

    def contains(self, x, y):
        c = self.corners
        return c[0][0] <= x <= c[3][0] and c[0][1] <= y <= c[3][1]

    def reflect_angle(self, x, y, theta):
        c = self.corners
        x_enter = not (c[0][0] <= x <= c[3][0])
        y_enter = not (c[0][1] <= y <= c[3][1])
        if x_enter and y_enter:
            return (theta + np.pi) % (2 * np.pi)
        elif x_enter:
            return (np.pi - theta) % (2 * np.pi)
        else:
            return (-theta) % (2 * np.pi)

    def draw(self, ax, color='gray', alpha=0.6):
        c = self.corners
        width  = c[3][0] - c[0][0]
        height = c[3][1] - c[0][1]
        rect = plt.Rectangle((c[0][0], c[0][1]), width, height,
                              color=color, alpha=alpha, zorder=2)
        ax.add_patch(rect)


class TriangleObstacle:
    def __init__(self, vertices):
        """vertices: [(x0,y0), (x1,y1), (x2,y2)] — any winding order"""
        self.vertices = [np.array(v, dtype=float) for v in vertices]

    def _signed_area(self, p, a, b):
        return (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0])

    def contains(self, x, y):
        p = np.array([x, y])
        v = self.vertices
        d0 = self._signed_area(p, v[0], v[1])
        d1 = self._signed_area(p, v[1], v[2])
        d2 = self._signed_area(p, v[2], v[0])
        has_neg = (d0 < 0) or (d1 < 0) or (d2 < 0)
        has_pos = (d0 > 0) or (d1 > 0) or (d2 > 0)
        return not (has_neg and has_pos)

    def _nearest_edge(self, x, y):
        """Return index of the edge whose segment is closest to (x, y)."""
        p = np.array([x, y])
        v = self.vertices
        min_dist, nearest = float('inf'), 0
        for i in range(3):
            a, b = v[i], v[(i + 1) % 3]
            ab = b - a
            t = np.clip(np.dot(p - a, ab) / np.dot(ab, ab), 0.0, 1.0)
            dist = np.linalg.norm(p - (a + t * ab))
            if dist < min_dist:
                min_dist, nearest = dist, i
        return nearest

    def reflect_angle(self, x, y, theta):
        """Reflect travel angle theta across the outward normal of the nearest edge."""
        i = self._nearest_edge(x, y)
        v = self.vertices
        edge = v[(i + 1) % 3] - v[i]
        edge_angle = np.arctan2(edge[1], edge[0])
        return (2 * edge_angle - theta) % (2 * np.pi)

    def draw(self, ax, color='gray', alpha=0.6):
        from matplotlib.patches import Polygon as MplPolygon
        coords = [v.tolist() for v in self.vertices]
        patch = MplPolygon(coords, closed=True, color=color, alpha=alpha, zorder=2)
        ax.add_patch(patch)


class AngleDiscretization:
    def __init__(self, m):
        self.m = m
        self.angles = 2 * np.pi * np.arange(m) / m

    def choose_new(self, probabilities, current_index):
        draw = random.uniform(0, 1)
        if draw < probabilities[0]:
            new_index = current_index - 1
        elif draw < probabilities[0] + probabilities[1]:
            new_index = current_index
        else:
            new_index = current_index + 1
        return int(new_index % self.m)

    def nearest_index(self, angle):
        angle = angle % (2 * np.pi)
        diffs = np.minimum(
            np.abs(self.angles - angle),
            2 * np.pi - np.abs(self.angles - angle)
        )
        return int(np.argmin(diffs))


class SpeedDiscretization:
    def __init__(self, g, v_max):
        self.g = g
        self.speeds = v_max * np.arange(g) / (g - 1)

    def choose_new(self, probabilities, current_index):
        draw = random.uniform(0, 1)
        if draw < probabilities[0]:
            new_index = max(current_index - 1, 0)
        elif draw < probabilities[0] + probabilities[1]:
            new_index = current_index
        else:
            new_index = min(current_index + 1, self.g - 1)
        return int(new_index)


class FieldOfView:
    def __init__(self, angle_lim=np.pi/4, radius=10):
        self.angle_lim = angle_lim
        self.radius = radius

    def contains(self, point, agent_angle, target_point):
        diff = np.array(target_point) - np.array(point)
        angle_to_other = np.arctan2(diff[1], diff[0])
        if angle_to_other < 0:
            angle_to_other += 2 * np.pi
        offset = abs(angle_to_other - agent_angle) + np.pi
        offset = (offset % (2 * np.pi)) - np.pi
        if abs(offset) > self.angle_lim:
            return False
        return np.sqrt(diff[0]**2 + diff[1]**2) <= self.radius

    @staticmethod
    def transition_probability(h, k, theta_i, angle_disc, alpha=1, e1=0.2, e2=0.6):
        theta_h = angle_disc.angles[h]
        theta_k = angle_disc.angles[k]

        def ccw(a, b):
            """True if angle a is counter-clockwise from angle b on the circle."""
            return 0 < (a - b) % (2 * np.pi) < np.pi

        k_ccw = ccw(theta_k, theta_h)
        i_ccw = ccw(theta_i, theta_h)

        if k_ccw and i_ccw:
            return [0, alpha*(1-e1-e2), alpha*(e1+e2)]
        elif k_ccw and not i_ccw:
            return [alpha*e2, alpha*(1-e1-e2), alpha*e1]
        elif not k_ccw and i_ccw:
            return [alpha*e1, alpha*(1-e1-e2), alpha*e2]
        else:
            return [alpha*(e1+e2), alpha*(1-e1-e2), 0]

    @staticmethod
    def transition_probability_speed(rho, alpha_speed=1):
        """
        rho         - local density in FOV (count / N)
        alpha_speed - interaction strength in [0, 1]; 1 = always change, 0 = never change
        Returns [p_slow, p_stay, p_fast]
        High density -> more likely to slow down; low density -> more likely to speed up.
        """
        p_down = alpha_speed * rho
        p_up   = alpha_speed * (1 - rho)
        p_stay = 1 - alpha_speed
        return [p_down, p_stay, p_up]

    def interactions(self, agent_idx, t, xs, ys, angle_indices, angle_disc, theta_target, N):
        cx, cy = xs[t, agent_idx], ys[t, agent_idx]
        h_idx = int(angle_indices[t, agent_idx])
        agent_angle = angle_disc.angles[h_idx]
        angle_probs = np.zeros(3, dtype=np.float32)
        count = 0
        for i in range(N):
            if i == agent_idx:
                continue
            if self.contains([cx, cy], agent_angle, [xs[t, i], ys[t, i]]):
                angle_probs += self.transition_probability(
                    h_idx, angle_indices[t, i],
                    theta_target, angle_disc
                )
                count += 1
        rho = count / N
        speed_probs = np.array(self.transition_probability_speed(rho), dtype=np.float32)
        if count > 0:
            return angle_probs / count, speed_probs
        else:
            # Option A: seek target individually when no neighbors in FOV
            i_ccw = 0 < (theta_target - agent_angle) % (2 * np.pi) < np.pi
            if i_ccw:
                return np.array([0.2, 0.2, 0.6], dtype=np.float32), speed_probs
            else:
                return np.array([0.6, 0.2, 0.2], dtype=np.float32), speed_probs

In [ ]:
class Simulation:
    def __init__(self, num_agents, boundary, exit_point, start_area, obstacle,
                 angle_disc, speed_discs, fov,
                 total_time, num_frames,
                 bypass_waypoints=None, waypoint_threshold=0.15,
                 max_exit_rate=None, type_proportions=None,
                 inflow_schedule=None, inflow_line=None):
        self.boundary = boundary
        self.exit_point = exit_point
        self.start_area = start_area
        self.obstacle = obstacle
        self.bypass_waypoints = bypass_waypoints
        self.waypoint_threshold = waypoint_threshold
        self.angle_disc = angle_disc
        self.num_types = len(speed_discs)
        self.type_proportions = type_proportions
        self.fov = fov
        self.max_exit_rate = max_exit_rate
        self.exit_credits = 0.0
        self.total_time = total_time
        self.num_frames = num_frames
        self.dt = total_time / num_frames  # seconds per frame
        # Scale speed_discs: input v_max is in m/s, convert to m/frame
        self.speed_discs = [SpeedDiscretization(sd.g, sd.speeds[-1] * self.dt)
                            for sd in speed_discs]
        self.inflow_line = inflow_line
        self._next_inflow = 0
        self.n = None
        self.xs = None
        self.ys = None
        self.angle_indices = None
        self.speed_indices = None
        self.active = None
        self.waypoint_phase = None
        self.agent_types = None

        # Build spawn plan from inflow_schedule (times in seconds, rate in agents/s)
        if inflow_schedule is not None:
            self._spawn_at = {}  # {frame_t: [wave_props_or_None, ...]}
            total_N = 0
            for entry in inflow_schedule:
                t_start_s, t_end_s, rate_s = entry[0], entry[1], entry[2]
                wave_props = entry[3] if len(entry) > 3 else None
                t_start_f = round(t_start_s / self.dt)
                t_end_f   = round(t_end_s   / self.dt)
                n_frames_wave = t_end_f - t_start_f
                total_wave = round(rate_s * (t_end_s - t_start_s))
                total_N += total_wave
                # Distribute total_wave evenly: first (total_wave % n_frames_wave)
                # frames get one extra agent so the total is always exact
                base  = total_wave // n_frames_wave
                extra = total_wave %  n_frames_wave
                for i in range(n_frames_wave):
                    count = base + (1 if i < extra else 0)
                    if count > 0:
                        tf = t_start_f + i
                        self._spawn_at.setdefault(tf, []).extend([wave_props] * count)
            self.N = total_N
        else:
            self._spawn_at = None
            self.N = num_agents

    def _sample_type(self, proportions=None):
        if proportions is None:
            proportions = self.type_proportions
        if proportions is None:
            proportions = [1 / self.num_types] * self.num_types
        proportions = np.array(proportions, dtype=float)
        proportions /= proportions.sum()
        return int(np.random.choice(self.num_types, p=proportions))

    def _initialize(self):
        N = self.N
        self.n = 1
        self.active = np.zeros((1, N)) if self._spawn_at is not None else np.ones((1, N))
        self.xs = np.ones((1, N)) * 1000
        self.ys = np.ones((1, N)) * 1000
        self.angle_indices = np.zeros((1, N), dtype=np.int32)
        self.speed_indices = np.zeros((1, N), dtype=np.int32)
        self.waypoint_phase = np.zeros(N, dtype=np.int32)
        self._next_inflow = 0

        if self._spawn_at is not None:
            self.agent_types = np.full(N, -1, dtype=np.int32)
        else:
            if self.type_proportions is None:
                proportions = [1 / self.num_types] * self.num_types
            else:
                proportions = self.type_proportions
            counts = [int(round(p * N)) for p in proportions]
            counts[-1] += N - sum(counts)
            self.agent_types = np.concatenate([
                np.full(count, k, dtype=np.int32) for k, count in enumerate(counts)
            ])
            random.shuffle(self.agent_types)

            c = self.start_area.corners
            x_min, x_max = c[0][0], c[1][0]
            y_min, y_max = c[0][1], c[2][1]
            for j in range(N):
                self.xs[0, j] = random.uniform(x_min, x_max)
                self.ys[0, j] = random.uniform(y_min, y_max)
                self.angle_indices[0, j] = random.randint(0, self.angle_disc.m)
                disc = self.speed_discs[self.agent_types[j]]
                self.speed_indices[0, j] = random.randint(0, disc.g)

    def _spawn_position(self, t, j):
        if self.inflow_line is not None:
            p0 = np.array(self.inflow_line[0], dtype=float)
            p1 = np.array(self.inflow_line[1], dtype=float)
        else:
            c = self.boundary.corners
            p0 = np.array([c[0][0], c[0][1]], dtype=float)
            p1 = np.array([c[0][0], c[2][1]], dtype=float)
        s = random.uniform(0, 1)
        pos = p0 + s * (p1 - p0)
        eps = 0.01
        bc = self.boundary.corners
        pos[0] = np.clip(pos[0], bc[0][0] + eps, bc[3][0] - eps)
        pos[1] = np.clip(pos[1], bc[0][1] + eps, bc[3][1] - eps)
        self.xs[t, j] = pos[0]
        self.ys[t, j] = pos[1]
        self.angle_indices[t, j] = random.randint(0, self.angle_disc.m)
        disc = self.speed_discs[self.agent_types[j]]
        self.speed_indices[t, j] = random.randint(0, disc.g)

    def _apply_inflow(self, t):
        for wave_props in self._spawn_at.get(t, []):
            if self._next_inflow >= self.N:
                return
            j = self._next_inflow
            self._next_inflow += 1
            self.agent_types[j] = self._sample_type(wave_props)
            self.active[t:, j] = 1
            self._spawn_position(t, j)

    def run(self, num_timesteps=None):
        if num_timesteps is None:
            num_timesteps = self.num_frames
        if self.xs is None:
            self._initialize()

        N = self.N
        t_start = self.n - 1

        self.xs = np.vstack([self.xs, np.ones((num_timesteps, N)) * 1000])
        self.ys = np.vstack([self.ys, np.ones((num_timesteps, N)) * 1000])
        self.angle_indices = np.vstack([self.angle_indices, np.zeros((num_timesteps, N), dtype=np.int32)])
        self.speed_indices = np.vstack([self.speed_indices, np.zeros((num_timesteps, N), dtype=np.int32)])
        self.active = np.vstack([self.active, np.tile(self.active[t_start], (num_timesteps, 1))])
        self.n += num_timesteps

        for t in range(t_start, self.n - 1):
            if self._spawn_at is not None:
                self._apply_inflow(t)
            if self.max_exit_rate is not None:
                self.exit_credits += self.max_exit_rate * self.dt
            for j in range(N):
                if self.active[t, j] == 0:
                    continue
                x, y = self.xs[t, j], self.ys[t, j]
                speed_disc = self.speed_discs[self.agent_types[j]]

                if self.bypass_waypoints and self.waypoint_phase[j] == 0:
                    dists = [np.sqrt((x - wp[0])**2 + (y - wp[1])**2)
                             for wp in self.bypass_waypoints]
                    best = int(np.argmin(dists))
                    if x > self.bypass_waypoints[0][0]:
                        self.waypoint_phase[j] = 1
                    wp = self.bypass_waypoints[best]
                    theta_target = np.arctan2(wp[1] - y, wp[0] - x) % (2 * np.pi)
                else:
                    theta_target = self.exit_point.angle_to(x, y)

                angle_probs, speed_probs = self.fov.interactions(
                    j, t, self.xs, self.ys,
                    self.angle_indices, self.angle_disc, theta_target, N
                )
                self.angle_indices[t+1, j] = self.angle_disc.choose_new(angle_probs, self.angle_indices[t, j])
                self.speed_indices[t+1, j] = speed_disc.choose_new(speed_probs, self.speed_indices[t, j])
                theta = self.angle_disc.angles[self.angle_indices[t, j]]
                v = speed_disc.speeds[self.speed_indices[t, j]]
                nx = x + v * np.cos(theta)
                ny = y + v * np.sin(theta)

                if self.obstacle != []:
                    if self.obstacle.contains(nx, ny):
                        reflected = self.obstacle.reflect_angle(x, y, theta)
                        self.angle_indices[t+1, j] = self.angle_disc.nearest_index(reflected)
                        nx, ny = x, y
                if not self.boundary.contains(nx, ny):
                    if self.exit_point.is_exiting(nx, ny):
                        if self.max_exit_rate is None or self.exit_credits >= 1:
                            self.active[t+1:, j] = 0
                            self.xs[t+1, j] = 1000
                            self.ys[t+1, j] = 1000
                            if self.max_exit_rate is not None:
                                self.exit_credits -= 1
                            continue
                        else:
                            self.xs[t+1, j] = x
                            self.ys[t+1, j] = y
                            continue
                    c = self.boundary.corners
                    x_hit = nx <= c[0][0] or nx >= c[3][0]
                    y_hit = ny <= c[0][1] or ny >= c[3][1]
                    if x_hit and y_hit:
                        reflected = theta + np.pi
                    elif x_hit:
                        reflected = np.pi - theta
                    else:
                        reflected = -theta
                    self.angle_indices[t+1, j] = self.angle_disc.nearest_index(reflected)
                    nx, ny = x, y

                self.xs[t+1, j] = nx
                self.ys[t+1, j] = ny

In [ ]:
class Visualizer:
    # Colors for agent speed types: index 0 = slowest, last index = fastest
    TYPE_COLORS = ['tab:red', 'tab:orange', 'tab:blue']
    TYPE_LABELS = ['slow', 'medium', 'fast']

    def __init__(self, simulation):
        self.sim = simulation
        self.fig = None
        self.ax = None
        self._agent_plots = []
        self._title = None
        self._anim = None

    def setup(self, figsize=(10, 6), margin=2):
        self.fig, self.ax = plt.subplots(figsize=figsize)
        c = self.sim.boundary.corners
        x_min, x_max = c[0][0], c[3][0]
        y_min, y_max = c[0][1], c[3][1]
        self.ax.set_xlim(x_min - margin, x_max + margin)
        self.ax.set_ylim(y_min - margin, y_max + margin)
        self.ax.set_aspect('equal')
        self.ax.set_xlabel('X-Coordinate')
        self.ax.set_ylabel('Y-Coordinate')
        self.sim.boundary.draw(self.ax, color='black')
        self.sim.exit_point.draw(self.ax, color='green')
        if self.sim._spawn_at is None:
            self.sim.start_area.draw(self.ax, color='blue')
        if self.sim.obstacle != []:
            self.sim.obstacle.draw(self.ax, color='gray', alpha=0.6)
        # Draw bypass waypoints
        for wp in self.sim.bypass_waypoints or []:
            self.ax.plot(*wp, 'x', color='orange', ms=8, markeredgewidth=2)
        self.ax.plot(*self.sim.exit_point.target, 'r.', ms=10)
        # Legend proxies
        if self.sim._spawn_at is None:
            self.ax.plot([], [], color='blue', label='start area')
        self.ax.plot([], [], color='green',  label='exit')
        self.ax.plot([], [], color='gray',   label='obstacle', linewidth=6, alpha=0.6)
        self.ax.plot([], [], 'x', color='orange', label='bypass waypoints', ms=8, markeredgewidth=2)
        for k in range(self.sim.num_types):
            color = self.TYPE_COLORS[k % len(self.TYPE_COLORS)]
            label = self.TYPE_LABELS[k] if k < len(self.TYPE_LABELS) else f'type {k}'
            self.ax.plot([], [], 'o', color=color, label=f'agent ({label})', ms=5)
        self.ax.legend(loc="upper right", fontsize='x-small')
        self._agent_plots = [
            self.ax.plot([], [], 'o',
                         color=self.TYPE_COLORS[self.sim.agent_types[i] % len(self.TYPE_COLORS)],
                         ms=5)[0]
            for i in range(self.sim.N)
        ]
        self._title = self.ax.set_title('')
        return self

    def _draw_frame(self, t):
        for i, plot in enumerate(self._agent_plots):
            plot.set_data([self.sim.xs[t, i]], [self.sim.ys[t, i]])
        self._title.set_text(f'Time = {t * self.sim.dt:.1f} s')
        return self._agent_plots

    def animate(self, num_timesteps=None, interval=50):
        if num_timesteps is None:
            frames = self.sim.n
        else:
            frames = num_timesteps
        self._anim = animation.FuncAnimation(
            self.fig, self._draw_frame,
            frames=frames, interval=interval, blit=True
        )
        return self._anim

    def save(self, filename, fps=40):
        writer = animation.PillowWriter(fps=fps, metadata=dict(artist='Me'), bitrate=1800)
        if self._anim is None:
            self.animate()
        self._anim.save(filename, writer=writer)

    def plot_pov(self, t, agent_idx, figsize=(8, 8), margin=2, zoom=None):
        """Visualize one agent's field of view at timestep t."""
        from matplotlib.patches import Wedge

        sim = self.sim
        if sim.active[t, agent_idx] == 0:
            print(f"Agent {agent_idx} has already exited by t={t}; nothing to show.")
            return None, None

        fig, ax = plt.subplots(figsize=figsize)

        sim.boundary.draw(ax, color='black')
        sim.exit_point.draw(ax, color='green')
        if sim.obstacle != []:
            sim.obstacle.draw(ax, color='gray', alpha=0.6)

        cx, cy = sim.xs[t, agent_idx], sim.ys[t, agent_idx]
        h_idx = int(sim.angle_indices[t, agent_idx])
        agent_angle = sim.angle_disc.angles[h_idx]

        heading_deg = np.degrees(agent_angle)
        angle_lim_deg = np.degrees(sim.fov.angle_lim)
        wedge = Wedge((cx, cy), sim.fov.radius,
                      heading_deg - angle_lim_deg, heading_deg + angle_lim_deg,
                      color='gold', alpha=0.3, zorder=1)
        ax.add_patch(wedge)

        arrow_len = sim.fov.radius * 0.3
        ax.annotate('', xy=(cx + arrow_len * np.cos(agent_angle), cy + arrow_len * np.sin(agent_angle)),
                     xytext=(cx, cy), arrowprops=dict(arrowstyle='->', color='black', lw=2), zorder=5)

        in_x, in_y, out_x, out_y = [], [], [], []
        for i in range(sim.N):
            if i == agent_idx or sim.active[t, i] == 0:
                continue
            px, py = sim.xs[t, i], sim.ys[t, i]
            if sim.fov.contains([cx, cy], agent_angle, [px, py]):
                in_x.append(px); in_y.append(py)
            else:
                out_x.append(px); out_y.append(py)

        ax.plot(out_x, out_y, 'o', color='lightgray', ms=5, label='outside FOV', zorder=2)
        ax.plot(in_x, in_y, 'o', color='red', ms=6, label='inside FOV', zorder=3)
        ax.plot([cx], [cy], '*',
                color=self.TYPE_COLORS[sim.agent_types[agent_idx] % len(self.TYPE_COLORS)],
                ms=18, label=f'agent {agent_idx}', zorder=6)

        if zoom is None:
            zoom = sim.fov.radius + margin
        ax.set_xlim(cx - zoom, cx + zoom)
        ax.set_ylim(cy - zoom, cy + zoom)
        ax.set_aspect('equal')
        ax.set_xlabel('X-Coordinate')
        ax.set_ylabel('Y-Coordinate')
        ax.set_title(f'Field of view — agent {agent_idx}, t={t * sim.dt:.1f}s')
        ax.legend(loc='upper right')
        return fig, ax


In [ ]:
boundary   = Boundary([[0, 0], [20, 0], [0, 10], [20, 10]])
exit_point = Exit([[17.5, 2.5], [20, 2.5], [17.5, 7.5], [20, 7.5]], target=[20, 5])
start_area = Boundary([[0, 0], [5, 0], [0, 10], [5, 10]])
obstacle   = Obstacle([[8.5, 3], [11.5, 3], [8.5, 7], [11.5, 7]])

bypass_waypoints = [
    (10, 8.5),  # upper bypass
    (10, 1.5),  # lower bypass
]

angle_disc = AngleDiscretization(32)
speed_disc_slow   = SpeedDiscretization(32, 0.6)   # m/s
speed_disc_medium = SpeedDiscretization(32, 1.2)   # m/s
speed_disc_fast   = SpeedDiscretization(32, 1.8)   # m/s
speed_discs = [speed_disc_slow, speed_disc_medium, speed_disc_fast]
fov = FieldOfView(angle_lim=np.pi/8, radius=10)

sim = Simulation(
    num_agents=200,
    boundary=boundary,
    exit_point=exit_point,
    start_area=start_area,
    obstacle=obstacle,
    bypass_waypoints=bypass_waypoints,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
)
sim.run()

In [ ]:
viz = Visualizer(sim)
viz.setup()
anim = viz.animate()
HTML(anim.to_jshtml())


In [ ]:
# Visualize one agent's field of view (POV) at a given timestep:
# the gold wedge is its FOV cone (angle_lim, radius), the arrow is its
# heading, red dots are other agents currently inside its FOV, gray dots
# are agents outside it.
viz = Visualizer(sim)
fig, _ = viz.plot_pov(t=30, agent_idx=90)

fig.savefig("FOV_Example.png")

## Two obstacles: square + triangle

In [ ]:
class MultiObstacle:
    """Wraps multiple obstacles so the Simulation sees a single obstacle interface."""
    def __init__(self, obstacles):
        self.obstacles = obstacles
        self._last_hit = None  # set by contains(), read by reflect_angle()

    def contains(self, x, y):
        for obs in self.obstacles:
            if obs.contains(x, y):
                self._last_hit = obs
                return True
        return False

    def reflect_angle(self, x, y, theta):
        obs = self._last_hit if self._last_hit is not None else self.obstacles[0]
        return obs.reflect_angle(x, y, theta)

    def draw(self, ax, color='gray', alpha=0.6):
        for obs in self.obstacles:
            obs.draw(ax, color=color, alpha=alpha)


In [ ]:
boundary2   = Boundary([[0, 0], [20, 0], [0, 10], [20, 10]])
exit_point2 = Exit([[17.5, 2.5], [20, 2.5], [17.5, 7.5], [20, 7.5]], target=[20, 5])
start_area2 = Boundary([[0, 0], [5, 0], [0, 10], [5, 10]])

sq = Obstacle([[8.5, 5], [11.5, 5], [8.5, 9], [11.5, 9]])
tri = TriangleObstacle([(8, 4.5), (8, 1.5), (12, 3)])
obstacle2 = MultiObstacle([sq, tri])

bypass_waypoints2 = [
    (10, 9.5),  # upper bypass
    (10, 0.5),  # lower bypass
]

angle_disc2 = AngleDiscretization(32)
speed_disc2_slow   = SpeedDiscretization(32, 0.06)   # m/s
speed_disc2_medium = SpeedDiscretization(32, 0.12)   # m/s
speed_disc2_fast   = SpeedDiscretization(32, 0.18)   # m/s
speed_discs2 = [speed_disc2_slow, speed_disc2_medium, speed_disc2_fast]
fov2 = FieldOfView(angle_lim=np.pi/8, radius=10)

sim2 = Simulation(
    num_agents=200,
    boundary=boundary2,
    exit_point=exit_point2,
    start_area=start_area2,
    obstacle=obstacle2,
    bypass_waypoints=bypass_waypoints2,
    angle_disc=angle_disc2,
    speed_discs=speed_discs2,
    fov=fov2,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    type_proportions=[0.5, 0.3, 0.2],
)
sim2.run()

viz2 = Visualizer(sim2)
viz2.setup()
anim2 = viz2.animate()
HTML(anim2.to_jshtml())

### Triangle rotated 180° (apex pointing left)

In [ ]:
boundary3   = Boundary([[0, 0], [20, 0], [0, 10], [20, 10]])
exit_point3 = Exit([[17.5, 2.5], [20, 2.5], [17.5, 7.5], [20, 7.5]], target=[20, 5])
start_area3 = Boundary([[0, 0], [5, 0], [0, 10], [5, 10]])

sq3 = Obstacle([[8.5, 5], [11.5, 5], [8.5, 9], [11.5, 9]])
tri3 = TriangleObstacle([(12, 1.5), (12, 4.5), (8, 3)])
obstacle3 = MultiObstacle([sq3, tri3])

bypass_waypoints3 = [
    (10, 9.5),
    (10, 0.5),
]

angle_disc3 = AngleDiscretization(32)
speed_disc3_slow   = SpeedDiscretization(32, 0.06)   # m/s
speed_disc3_medium = SpeedDiscretization(32, 0.12)   # m/s
speed_disc3_fast   = SpeedDiscretization(32, 0.18)   # m/s
speed_discs3 = [speed_disc3_slow, speed_disc3_medium, speed_disc3_fast]
fov3 = FieldOfView(angle_lim=np.pi/8, radius=10)

sim3 = Simulation(
    num_agents=200,
    boundary=boundary3,
    exit_point=exit_point3,
    start_area=start_area3,
    obstacle=obstacle3,
    bypass_waypoints=bypass_waypoints3,
    angle_disc=angle_disc3,
    speed_discs=speed_discs3,
    fov=fov3,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    type_proportions=[0.5, 0.3, 0.2],
)
sim3.run()

viz3 = Visualizer(sim3)
viz3.setup()
anim3 = viz3.animate()
HTML(anim3.to_jshtml())

In [ ]:
boundary4   = Boundary([[0, 0], [20, 0], [0, 10], [20, 10]])
exit_point4 = Exit([[17.5, 2.5], [20, 2.5], [17.5, 7.5], [20, 7.5]], target=[20, 5])
start_area4 = Boundary([[0, 0], [5, 0], [0, 10], [5, 10]])

tri4_a = TriangleObstacle([(12, 8.5), (12, 5.5), (8, 7)])
tri4_b = TriangleObstacle([(12, 1.5), (12, 4.5), (8, 3)])
obstacle4 = MultiObstacle([tri4_a, tri4_b])

bypass_waypoints4 = [
    (10, 9.5),
    (10, 0.5),
]

angle_disc4 = AngleDiscretization(32)
speed_disc4_slow   = SpeedDiscretization(32, 0.06)   # m/s
speed_disc4_medium = SpeedDiscretization(32, 0.12)   # m/s
speed_disc4_fast   = SpeedDiscretization(32, 0.18)   # m/s
speed_discs4 = [speed_disc4_slow, speed_disc4_medium, speed_disc4_fast]
fov4 = FieldOfView(angle_lim=np.pi/8, radius=10)

sim4 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obstacle4,
    bypass_waypoints=bypass_waypoints4,
    angle_disc=angle_disc4,
    speed_discs=speed_discs4,
    fov=fov4,
    total_time=50,
    num_frames=3,
    waypoint_threshold=0.15,
    type_proportions=[0.5, 0.3, 0.2],
)
sim4.run()
viz4 = Visualizer(sim4)
viz4.setup()
anim4 = viz4.animate()
HTML(anim4.to_jshtml())

## Wide Bottleneck Border

In [ ]:
#Initialising Environment
boundary4   = Boundary([[0, 0], [20, 0], [0, 10], [20, 10]])
exit_point4 = Exit([[17.5, 2.5], [20, 2.5], [17.5, 7.5], [20, 7.5]], target=[20, 5])
start_area4 = Boundary([[0, 0], [5, 0], [0, 10], [5, 10]])
def make_rect(cx, cy, w, h):
    return [[cx-w/2, cy-h/2], [cx+w/2, cy-h/2], [cx-w/2, cy+h/2], [cx+w/2, cy+h/2]]
sq1 = make_rect(10.0, 9.0, 8.0, 2.0)
sq2 = make_rect(10.0, 1.0, 8.0, 2.0)
ob_sq1 = Obstacle(sq1)
ob_sq2 = Obstacle(sq2)
obstacle_Wide_Bottleneck = MultiObstacle([ob_sq1, ob_sq2])

#Initialising Simulation
angle_disc = AngleDiscretization(32)
speed_disc_slow   = SpeedDiscretization(32, 0.6)   # m/s
speed_disc_medium = SpeedDiscretization(32, 1.2)   # m/s
speed_disc_fast   = SpeedDiscretization(32, 1.8)   # m/s
speed_discs = [speed_disc_slow, speed_disc_medium, speed_disc_fast]
fov = FieldOfView(angle_lim=np.pi/8, radius=10)

sim_Wide_Bottleneck = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obstacle_Wide_Bottleneck,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
)

#Initialising Animation
sim_Wide_Bottleneck.run()
viz_Wide_Bottleneck = Visualizer(sim_Wide_Bottleneck)
viz_Wide_Bottleneck.setup()
anim_Wide_Bottleneck = viz_Wide_Bottleneck.animate()
HTML(anim_Wide_Bottleneck.to_jshtml())

## Open Road

In [ ]:
speed_disc_slow   = SpeedDiscretization(32, 0.6)   # m/s
speed_disc_medium = SpeedDiscretization(32, 1.2)   # m/s
speed_disc_fast   = SpeedDiscretization(32, 1.8)   # m/s
speed_discs = [speed_disc_slow, speed_disc_medium, speed_disc_fast]
sim_Open_Road = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=[],
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
)

#Initialising Animation
sim_Open_Road.run()
viz_Open_Road = Visualizer(sim_Open_Road)
viz_Open_Road.setup()
anim_Open_Road = viz_Open_Road.animate()
HTML(anim_Open_Road.to_jshtml())

## Strong Bottleneck

In [ ]:
sim_Strong_Bottleneck = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=[],
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
)

#Initialising Animation
sim_Strong_Bottleneck.run()
viz_Strong_Bottleneck = Visualizer(sim_Strong_Bottleneck)
viz_Strong_Bottleneck.setup()
anim_Strong_Bottleneck = viz_Strong_Bottleneck.animate()
HTML(anim_Strong_Bottleneck.to_jshtml())

## Stupid Validation

In [ ]:
tri_1 = TriangleObstacle([(10, 5), (10, 1.5), (7, 3)])
tri_2 = TriangleObstacle([(10, 8.5), (10, 5), (7, 7)])
little_square = Obstacle([[9,4],[10,4],[9,6],[10,6]])

obs_stupid = MultiObstacle([little_square, tri_1, tri_2])

sim_Stupid_Validation = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_stupid,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule=[(0, 16.7, 10)],  # 0–16.7s at 60 agents/s = 1000 agents
    inflow_line=((0, 2), (0, 8))
)

#Initialising Animation
sim_Stupid_Validation.run()
viz_Stupid_Validation = Visualizer(sim_Stupid_Validation)
viz_Stupid_Validation.setup()
anim_Stupid_Validation = viz_Stupid_Validation.animate()
HTML(anim_Stupid_Validation.to_jshtml())

In [ ]:
#BAD STUPIC VALIDAtION. PEOPLE LEAK THROUGH

tri_1 = TriangleObstacle([(10, 5), (10, 1.5), (7, 3)])
tri_2 = TriangleObstacle([(10, 8.5), (10, 5), (7, 7)])

obs_stupid = MultiObstacle([tri_1, tri_2])

sim_Stupid_Validation = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_stupid,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
)

#Initialising Animation
sim_Stupid_Validation.run()
viz_Stupid_Validation = Visualizer(sim_Stupid_Validation)
viz_Stupid_Validation.setup()
anim_Stupid_Validation = viz_Stupid_Validation.animate()
HTML(anim_Stupid_Validation.to_jshtml())

## Thick Bottleneck

In [ ]:
#Initialising Environment
boundary4   = Boundary([[0, 0], [20, 0], [0, 10], [20, 10]])
exit_point4 = Exit([[17.5, 2.5], [20, 2.5], [17.5, 7.5], [20, 7.5]], target=[20, 5])
start_area4 = Boundary([[0, 0], [5, 0], [0, 10], [5, 10]])

sq1 = make_rect(10.0, 8, 4, 4)
sq2 = make_rect(10.0, 2, 4, 4)
ob_sq1 = Obstacle(sq1)
ob_sq2 = Obstacle(sq2)
obstacle_Wide_Bottleneck = MultiObstacle([ob_sq1, ob_sq2])

#Initialising Simulation
angle_disc = AngleDiscretization(32)
speed_disc_slow   = SpeedDiscretization(32, 0.6)   # m/s
speed_disc_medium = SpeedDiscretization(32, 1.2)   # m/s
speed_disc_fast   = SpeedDiscretization(32, 1.8)   # m/s
speed_discs = [speed_disc_slow, speed_disc_medium, speed_disc_fast]
fov = FieldOfView(angle_lim=np.pi/8, radius=10)

sim_Wide_Bottleneck = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obstacle_Wide_Bottleneck,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
)

#Initialising Animation
sim_Wide_Bottleneck.run()
viz_Wide_Bottleneck = Visualizer(sim_Wide_Bottleneck)
viz_Wide_Bottleneck.setup()
anim_Wide_Bottleneck = viz_Wide_Bottleneck.animate()
HTML(anim_Wide_Bottleneck.to_jshtml())

## 3 Obstacles

In [ ]:
SAVE_GIF = True


tri_1 = TriangleObstacle([(14, 2), (17, 2), (15.5, 5)])
rec_1 = Obstacle(make_rect(3, 7, 2, 1.5))
rec_2 = Obstacle(make_rect(10, 5, 4, 4))



obs_triple = MultiObstacle([tri_1, rec_1, rec_2])
speed_disc_slow   = SpeedDiscretization(32, 0.8)   # m/s
speed_disc_medium = SpeedDiscretization(32, 1.3)   # m/s
speed_disc_fast   = SpeedDiscretization(32, 2.0)   # m/s
speed_discs = [speed_disc_slow, speed_disc_medium, speed_disc_fast]

Total_Frames = 300

sim_triple = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_triple,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =[(0, 7, 4,[0,0,1]),
                     (7,10,8,[0,1,0]),
                     (10,15,6,[1,0,0])
    ],
    inflow_line=[(0,2),(0,8)]
)

#Initialising Animation
sim_triple.run()
viz_triple = Visualizer(sim_triple)
viz_triple.setup()
anim_triple = viz_triple.animate()
HTML(anim_triple.to_jshtml())

if SAVE_GIF:
    viz_triple.save("Test1235.gif", fps=Total_Frames//10)

## Thick Middle Block

## Three Corridors Left

## Triangular Point left

In [ ]:
tri = TriangleObstacle([(12, 7), (12, 3), (8, 5)])

sim_Triang_Left = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=tri,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
)

#Initialising Animation
sim_Triang_Left.run()
viz_Triang_Left = Visualizer(sim_Triang_Left)
viz_Triang_Left.setup()
anim_Triang_Left = viz_Triang_Left.animate()
HTML(anim_Triang_Left.to_jshtml())

## Triangular Point Right

In [ ]:
tri_right = TriangleObstacle([(8, 3), (8, 7), (12, 5)])

sim_Triang_Right = Simulation(
    num_agents=90,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=tri_right,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=300,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
)

#Initialising Animation
sim_Triang_Right.run()
viz_Triang_Right = Visualizer(sim_Triang_Right)
viz_Triang_Right.setup()
anim_Triang_Right = viz_Triang_Right.animate()
HTML(anim_Triang_Right.to_jshtml())

## Central Divider

## Animations for the Presentation

In [ ]:
Exit_Rate = 1
Total_Frames = 300
fps = Total_Frames//10
inflow_line=[(0,2),(0,8)]
speed_disc_slow   = SpeedDiscretization(32, 0.8)   # m/s
speed_disc_medium = SpeedDiscretization(32, 1.3)   # m/s
speed_disc_fast   = SpeedDiscretization(32, 2.0)   # m/s
speed_discs_tree_obstacles = [speed_disc_slow, speed_disc_medium, speed_disc_fast]
#3 Groups Comp
#No Obstacles
obs_no_obs = []
#Mass

#Fast in front

#Slow in Front

#Three Obstacles
tri_1 = TriangleObstacle([(14, 2), (17, 2), (15.5, 5)])
rec_1 = Obstacle(make_rect(3, 7, 2, 1.5))
rec_2 = Obstacle(make_rect(10, 5, 4, 4))

obs_triple = MultiObstacle([tri_1, rec_1, rec_2])

#Mass
inflow = [(0,4,20,[0.5,0.3,0.2]),
          (8,12,35,[0.5,0.3,0.2]),
          (18,20,50,[0.5,0.3,0.2])]  #(start,end,rate,composition)
#Fast in Front
inflow_FIF = [(0,4,20,[0,0,1]),
          (8,12,35,[0,1,0]),
          (18,20,50,[1,0,0])] 
#Slow in Front
inflow_SIF = [(0,4,20,[1,0,0]),
          (8,12,35,[0,1,0]),
          (18,20,50,[0,0,1])] 
#Group Comp
inflow_group = [(0,1,300,[0.5,0.3,0.2])]
#No Obstacle
obs_no_obs = []
#Triangle_Left
tri_left = TriangleObstacle([(12, 7), (12, 3), (8, 5)])
#Bottleneck
sq1 = make_rect(10.0, 8, 4, 4)
sq2 = make_rect(10.0, 2, 4, 4)
ob_sq1 = Obstacle(sq1)
ob_sq2 = Obstacle(sq2)
obstacle_Wide_Bottleneck = MultiObstacle([ob_sq1, ob_sq2])
#Stupid Validation
tri_1 = TriangleObstacle([(10, 5), (10, 1.5), (7, 3)])
tri_2 = TriangleObstacle([(10, 8.5), (10, 5), (7, 7)])
little_square = Obstacle([[9,4],[10,4],[9,6],[10,6]])

obs_stupid = MultiObstacle([little_square, tri_1, tri_2])


In [ ]:
sim_A1 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_no_obs,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=Exit_Rate,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow,
    inflow_line=[(0,2),(0,8)]
)

sim_A2 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_no_obs,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=Exit_Rate,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_FIF,
    inflow_line=[(0,2),(0,8)]
)
sim_A3 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_no_obs,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=Exit_Rate,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_SIF,
    inflow_line=[(0,2),(0,8)]
)

sim_B1 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_triple,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow,
    inflow_line=[(0,2),(0,8)]
)

sim_B2 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_triple,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_FIF,
    inflow_line=[(0,2),(0,8)]
)

sim_B3 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_triple,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_SIF,
    inflow_line=[(0,2),(0,8)]
)

sim_C1 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_no_obs,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_group,
    inflow_line=[(0,2),(0,8)]
)

sim_C2 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=tri_left,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_group,
    inflow_line=[(0,2),(0,8)]
)

sim_C3 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obstacle_Wide_Bottleneck,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_group,
    inflow_line=[(0,2),(0,8)]
)

sim_C4 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_stupid,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=50,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=1,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_group,
    inflow_line=[(0,2),(0,8)]
)

In [ ]:
sim_A1.run()
viz_A1 = Visualizer(sim_A1)
viz_A1.setup()
anim_A1 = viz_A1.animate()
HTML(anim_A1.to_jshtml())


In [ ]:
def plot_snapshots(sim, times, ncols=2, ms=6):
    """Particle snapshots at the given times (seconds). Dots coloured by
    agent type, with boundary, exit and obstacles drawn."""
    C, L = Visualizer.TYPE_COLORS, Visualizer.TYPE_LABELS
    c = sim.boundary.corners
    x0, x1, y0, y1 = c[0][0], c[3][0], c[0][1], c[3][1]
    frames = [min(int(round(t / sim.dt)), sim.n - 1) for t in times]

    nrows = int(np.ceil(len(frames) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3 * nrows), squeeze=False)
    axes = axes.ravel()
    for ax, f in zip(axes, frames):
        sim.boundary.draw(ax, color='black')
        sim.exit_point.draw(ax, color='green')
        if sim.obstacle != []:
            sim.obstacle.draw(ax, color='gray', alpha=0.6)
        active = sim.active[f] == 1                     # exclude exited / not-yet-spawned
        for k in range(sim.num_types):
            m = active & (sim.agent_types == k)
            ax.plot(sim.xs[f][m], sim.ys[f][m], 'o', color=C[k], ms=ms,
                    label=L[k] if ax is axes[0] else None)
        ax.set_xlim(x0 - 1, x1 + 1); ax.set_ylim(y0 - 1, y1 + 1)
        ax.set_aspect('equal'); ax.set_title(f't = {f * sim.dt:.2f} s')
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    for ax in axes[len(frames):]:
        ax.axis('off')
    axes[0].legend(loc='upper right', fontsize='small')
    fig.tight_layout()
    return fig

In [ ]:
fig1 = plot_snapshots(sim_C1, times=[3, 15, 35, 50])   # figure 1
fig1.savefig("figure1_snapshots.png", dpi=150)

fig3 = plot_snapshots(sim_B1, times=[3, 15, 35, 50])      # figure 3
fig3.savefig("figure3_snapshots.png", dpi=150)

In [ ]:
sim_A2.run()
viz_A2 = Visualizer(sim_A2)
viz_A2.setup()
anim_A2 = viz_A2.animate()
HTML(anim_A2.to_jshtml())


In [ ]:
sim_A3.run()
viz_A3 = Visualizer(sim_A3)
viz_A3.setup()
anim_A3 = viz_A3.animate()
HTML(anim_A3.to_jshtml())


In [ ]:
sim_B1.run()
viz_B1 = Visualizer(sim_B1)
viz_B1.setup()
anim_B1 = viz_B1.animate()
HTML(anim_B1.to_jshtml())


In [ ]:
sim_B2.run()
viz_B2 = Visualizer(sim_B2)
viz_B2.setup()
anim_B2 = viz_B2.animate()
HTML(anim_B2.to_jshtml())

In [ ]:
sim_B3.run()
viz_B3 = Visualizer(sim_B3)
viz_B3.setup()
anim_B3 = viz_B3.animate()
HTML(anim_B3.to_jshtml())

In [ ]:
sim_C1.run()
viz_C1 = Visualizer(sim_C1)
viz_C1.setup()
anim_C1 = viz_C1.animate()
HTML(anim_C1.to_jshtml())

In [ ]:
sim_C2.run()
viz_C2 = Visualizer(sim_C2)
viz_C2.setup()
anim_C2 = viz_C2.animate()
HTML(anim_C2.to_jshtml())

In [ ]:
sim_C3.run()
viz_C3 = Visualizer(sim_C3)
viz_C3.setup()
anim_C3 = viz_C3.animate()
HTML(anim_C3.to_jshtml())

In [ ]:
sim_C4.run()
viz_C4 = Visualizer(sim_C4)
viz_C4.setup()
anim_C4 = viz_C4.animate()
HTML(anim_C4.to_jshtml())

In [ ]:
viz_A1.save("A1_3_Groups_No_Obs_Mass.gif", fps=Total_Frames//10)
viz_A2.save("A2_3_Groups_No_Obs_FIF.gif", fps=Total_Frames//10)
viz_A3.save("A3_3_Groups_No_Obs_SIF.gif", fps=Total_Frames//10)
viz_B1.save("B1_3_Groups_3_Obs_Mass.gif", fps=Total_Frames//10)
viz_B2.save("B2_3_Groups_3_Obs_FIF.gif", fps=Total_Frames//10)
viz_B3.save("B3_3_Groups_3_Obs_SIF.gif", fps=Total_Frames//10)
viz_C1.save("C1_1_Group_No_Obs.gif", fps=Total_Frames//10)
viz_C2.save("C2_1_Group_TriL.gif", fps=Total_Frames//10)
viz_C3.save("C3_1_Group_Bottleneck.gif", fps=Total_Frames//10)
viz_C4.save("C4_1_Group_Stupid_Validation.gif", fps=Total_Frames//10)

In [ ]:
def conservation_curves(sim):
    active = sim.active                      # (T, N): 1 if agent inside at frame t
    T, N = active.shape
    types = sim.agent_types
    t = np.arange(T) * sim.dt

    inside_total = active.sum(axis=1)
    inside_group = np.array([active[:, types == k].sum(axis=1)
                             for k in range(sim.num_types)])

    entered = np.zeros((T, N)); exited = np.zeros((T, N))
    for j in range(N):
        col = active[:, j]
        ones = np.where(col == 1)[0]
        if len(ones) == 0:
            continue                         # agent never spawned
        t_in = ones[0]; entered[t_in:, j] = 1
        after = np.where(col[t_in:] == 0)[0] # first 0 after entering = its exit
        if len(after):
            exited[t_in + after[0]:, j] = 1

    cum_entered_g = np.array([entered[:, types == k].sum(axis=1) for k in range(sim.num_types)])
    cum_exited_g  = np.array([exited[:,  types == k].sum(axis=1) for k in range(sim.num_types)])
    balance_error = inside_total - (cum_entered_g.sum(0) - cum_exited_g.sum(0))
    return dict(t=t, inside_total=inside_total, inside_group=inside_group,
                cum_entered_g=cum_entered_g, cum_exited_g=cum_exited_g,
                balance_error=balance_error)


def plot_conservation(sim, title=""):
    d = conservation_curves(sim)
    C, L = Visualizer.TYPE_COLORS, Visualizer.TYPE_LABELS   # reuse your colours
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(d['t'], d['inside_total'], 'k-', label='total in domain')
    for k in range(sim.num_types):
        ax1.plot(d['t'], d['inside_group'][k], color=C[k], label=L[k])
    ax1.set_xlabel('t [s]'); ax1.set_ylabel('pedestrians in domain')
    ax1.set_title('mass currently inside the domain'); ax1.legend(fontsize='small')

    for k in range(sim.num_types):
        ax2.plot(d['t'], d['cum_entered_g'][k], color=C[k], label=f'{L[k]} in')
        ax2.plot(d['t'], d['cum_exited_g'][k],  color=C[k], ls='--', label=f'{L[k]} out')
    ax2.set_xlabel('t [s]'); ax2.set_ylabel('cumulative pedestrians')
    ax2.set_title('cumulative entered (—) vs exited (- -)'); ax2.legend(fontsize='small', ncol=3)
    ax2b = ax2.twinx()
    ax2b.plot(d['t'], d['balance_error'], color='gray', lw=0.8, alpha=0.6)
    ax2b.set_ylabel('balance error [ped]')

    fig.suptitle(title); fig.tight_layout()
    return fig, d

In [ ]:
fig, d = plot_conservation(sim_A2, "Fast-in-front")
# fig.savefig("conservation_FIF.png", dpi=150)
print("max |balance error| =", abs(d['balance_error']).max())   # -> 0.0

In [ ]:
fig, d = plot_conservation(sim_A3, "Slow-in-front")
# fig.savefig("conservation_FIF.png", dpi=150)
print("max |balance error| =", abs(d['balance_error']).max())   # -> 0.0

In [ ]:
Exit_Rate=2
Total_Frames = 450
sim_A2 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_no_obs,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=75,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=Exit_Rate,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_FIF,
    inflow_line=[(0,2),(0,8)]
)
sim_A3 = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_no_obs,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=75,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=Exit_Rate,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_SIF,
    inflow_line=[(0,2),(0,8)]
)

In [ ]:
sim_A2.run()
viz_A2 = Visualizer(sim_A2)
viz_A2.setup()
anim_A2 = viz_A2.animate()
HTML(anim_A2.to_jshtml())


In [ ]:
sim_A3.run()
viz_A3 = Visualizer(sim_A3)
viz_A3.setup()
anim_A3 = viz_A3.animate()
HTML(anim_A3.to_jshtml())


In [ ]:
Exit_Rate=5
Total_Frames = 450
sim_A2B = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_no_obs,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=75,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=Exit_Rate,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_FIF,
    inflow_line=[(0,2),(0,8)]
)
sim_A3B = Simulation(
    num_agents=200,
    boundary=boundary4,
    exit_point=exit_point4,
    start_area=start_area4,
    obstacle=obs_no_obs,
    angle_disc=angle_disc,
    speed_discs=speed_discs,
    fov=fov,
    total_time=75,
    num_frames=Total_Frames,
    waypoint_threshold=0.15,
    max_exit_rate=Exit_Rate,
    type_proportions=[0.5, 0.3, 0.2],
    inflow_schedule =inflow_SIF,
    inflow_line=[(0,2),(0,8)]
)

In [ ]:
sim_A2B.run()
viz_A2B = Visualizer(sim_A2B)
viz_A2B.setup()
anim_A2B = viz_A2B.animate()
HTML(anim_A2B.to_jshtml())


In [ ]:
sim_A3B.run()
viz_A3B = Visualizer(sim_A3B)
viz_A3B.setup()
anim_A3B = viz_A3B.animate()
HTML(anim_A3B.to_jshtml())


In [ ]:
fig, d = plot_conservation(sim_A2B, "Slow-in-front")
# fig.savefig("conservation_FIF.png", dpi=150)
print("max |balance error| =", abs(d['balance_error']).max())   # -> 0.0

In [ ]:
fig, d = plot_conservation(sim_A3B, "Slow-in-front")
# fig.savefig("conservation_FIF.png", dpi=150)
print("max |balance error| =", abs(d['balance_error']).max())   # -> 0.0